In [5]:
import pandas as pd
import numpy as np

# load doordash_earnings into df
sheet_id_earnings = "1Ca2KTWz0kxIWHgUxjw0gi97hZtqPOL0CnNh1XARyJZM"
earnings_sheet_name = "doordash_earnings"
url_earnings = f"https://docs.google.com/spreadsheets/d/{sheet_id_earnings}/gviz/tq?tqx=out:csv&sheet={earnings_sheet_name}"
df = pd.read_csv(url_earnings)

# load car_data into a separate df
sheet_id_car_data = "11fpKpHZ-A-4s37lHE-0M8ADkhcvcjT1KaT-60xwvF1c"
car_data_sheet_name = "car_data"
url_car_data = f"https://docs.google.com/spreadsheets/d/{sheet_id_car_data}/gviz/tq?tqx=out:csv&sheet={car_data_sheet_name}"
car_df = pd.read_csv(url_car_data)

# overall null check
null_summary = pd.DataFrame({
    "null_count": df.isnull().sum(),
    "null_pct":   (df.isnull().sum() / len(df) * 100).round(2),
    "dtype":      df.dtypes
}).sort_values("null_count", ascending=False)

print(f"Dataset shape: {df.shape}")
print(f"Total missing values: {df.isnull().sum().sum()}\n")

if df.isnull().sum().sum() == 0:
    print("All columns clean — no nulls found.")
else:
    print(null_summary[null_summary["null_count"] > 0].to_string())

# display all columns to help identify correct boolean column names
print("\nDataFrame columns:")
print(df.columns.tolist())

# boolean column verification
print("\n── Boolean Columns ─────────────────────────────────────────────────────")
bool_cols = ["uses_fuel", "has_peak_pay", "has_challenge_bonus"]
for col in bool_cols:
    if col in df.columns:
        print(f"{col}: unique values = {df[col].unique()}  |  nulls = {df[col].isnull().sum()}")
    else:
        print(f"Warning: Column '{col}' not found in DataFrame.")

# null Check
print(f"Total missing values: {df.isnull().sum().sum()}")
print(f"Dataset shape: {df.shape}\n")

print(df[df["vehicle_est_mpg"].isnull()]["vehicle_class"].value_counts())

# Fill nulls
df["vehicle_est_mpg"] = df["vehicle_est_mpg"].fillna(
    df["vehicle_class"].map({"Electric": 94, "Bicycle": 0})
)


# Classify each car_data row into DoorDash vehicle_class
# Strategy:
#   - EVs   → fuel_type: 'electricity'
#   - Hybrid → fuel_type: 'gas' and combination_mpg >= 35 (captures all labeled
#              hybrids in this dataset: Insight, Corolla Hybrid, Sportage Hybrid,
#              Tucson Hybrid, Volt; excludes efficient-but-not-hybrid gas cars)
#   - Other: mapped from EPA class string

EPA_CLASS_MAP = {
    'compact car':                    'Compact/Midsize',
    'midsize car':                    'Compact/Midsize',
    'large car':                      'Compact/Midsize',
    'subcompact car':                 'Compact/Midsize',
    'minicompact car':                'Compact/Midsize',
    'two seater':                     'Compact/Midsize',
    'small station wagon':            'Compact/Midsize',
    'midsize station wagon':          'Compact/Midsize',
    'small sport utility vehicle':    'SUV',
    'standard sport utility vehicle': 'SUV',
    'minivan':                        'SUV',
    'standard pickup truck':          'Truck',
    'small pickup truck':             'Truck',
}

HYBRID_MPG_THRESHOLD = 35  # mpg — all true hybrids in this dataset are >= 35

def assign_vehicle_class(row):
    if row['fuel_type'] == 'electricity':
        return 'Electric'
    if row['fuel_type'] == 'gas' and row['combination_mpg'] >= HYBRID_MPG_THRESHOLD:
        return 'Hybrid'
    return EPA_CLASS_MAP.get(row['class'], 'Compact/Midsize')  # fallback

car_df['vehicle_class'] = car_df.apply(assign_vehicle_class, axis=1)

# build MPG lookup per vehicle_class
# For Electric: use EPA MPGe (miles per gallon equivalent) — not in car_data
# For Motorcycle/Bicycle: static lookup (not in car_data at all)

# From car_data (computed)
mpg_from_data = (
    car_df[car_df['vehicle_class'].isin(['Compact/Midsize', 'SUV', 'Truck', 'Hybrid', 'Electric'])]
    .groupby('vehicle_class')['combination_mpg']
    .median()
)

# Static overrides for sparse/missing classes
STATIC_MPG = {
    'Electric':   94,    # EPA avg MPGe for BEVs (2023); gasoline-equivalent
    'Motorcycle': 55,    # NHTSA estimate for mid-size commuter motorcycle
    'Bicycle':    None,  # No fuel cost
}

# Merge: static overrides take priority for Electric (only 2 rows, unrepresentative)
mpg_lookup = mpg_from_data.to_dict()
mpg_lookup.update(STATIC_MPG)

print("=== Final MPG lookup table ===")
for cls, mpg in sorted(mpg_lookup.items()):
    source = "static" if cls in STATIC_MPG else f"car_data median (n={car_df[car_df['vehicle_class']==cls].shape[0]})"
    print(f"  {cls:<18} {str(mpg):<8} ({source})")

# Apply to DoorDash earnings

df['resolved_mpg'] = df['vehicle_class'].map(mpg_lookup)

# Bicycle: mpg doesn't matter. flag for cost calc
df['uses_fuel'] = df['vehicle_type'] != 'Bicycle'

# Boolean Column Verification
bool_cols = ["uses_fuel", "has_peak_pay", "has_challenge_bonus"]

for col in bool_cols:
    unique_vals = df[col].unique()
    null_count  = df[col].isnull().sum()
    print(f"{col}: unique values = {unique_vals}  |  nulls = {null_count}")

# gas prices (EIA by census region)
gas_prices = {
    'Pacific':   4.80,
    'Northeast': 4.05,
    'Midwest':   3.60,
    'South':     3.25,
    'Southwest': 3.50,
    'Mountain':  3.65,
}
df['avg_gas_price'] = df['census_region'].map(gas_prices)

# compute gas cost
df['gas_cost_usd'] = (
    df['total_miles_driven'] / df['resolved_mpg']
) * df['avg_gas_price']
df.loc[df['vehicle_type'] == 'Bicycle', 'gas_cost_usd'] = 0.0

# net earnings
df['gross_earnings_usd'] = df['doordash_pay_usd'] + df['gross_tips_usd']
df['net_earnings_usd']   = df['gross_earnings_usd'] - df['gas_cost_usd']
df['net_hourly_rate']    = df['net_earnings_usd'] / df['hours_worked']

df.to_csv('merged_doordoor_roi.csv', index=False)
display(df.head(10))

Dataset shape: (2000, 28)
Total missing values: 168

                 null_count  null_pct    dtype
vehicle_est_mpg         168       8.4  float64

DataFrame columns:
['session_id', 'region', 'census_region', 'time_slot', 'driver_experience_tier', 'vehicle_type', 'vehicle_class', 'vehicle_est_mpg', 'hours_worked', 'active_hours', 'inactive_hours', 'active_ratio_pct', 'total_deliveries', 'deliveries_per_hour', 'avg_delivery_duration_min', 'acceptance_rate_pct', 'total_miles_driven', 'avg_miles_per_delivery', 'doordash_pay_usd', 'gross_base_pay_usd', 'gross_peak_pay_usd', 'challenge_bonus_usd', 'base_pay_per_delivery', 'has_peak_pay', 'has_challenge_bonus', 'gross_tips_usd', 'tip_per_delivery', 'tip_rate_pct']

── Boolean Columns ─────────────────────────────────────────────────────
has_peak_pay: unique values = [ True False]  |  nulls = 0
has_challenge_bonus: unique values = [False  True]  |  nulls = 0
Total missing values: 168
Dataset shape: (2000, 28)

vehicle_class
Electric    118
Bi

,session_id,region,census_region,time_slot,driver_experience_tier,vehicle_type,vehicle_class,vehicle_est_mpg,hours_worked,active_hours,...,gross_tips_usd,tip_per_delivery,tip_rate_pct,resolved_mpg,uses_fuel,avg_gas_price,gas_cost_usd,gross_earnings_usd,net_earnings_usd,net_hourly_rate
0,1,"Houston, TX",South,Weekday Midday (2pm-5pm),Experienced (1-2 yr),SUV,SUV,24.0,3.43,2.86,...,19.75,3.95,100,23.0,True,3.25,4.561304,37.90,33.338696,9.719736
1,2,"Miami, FL",South,Weekday Lunch (11am-2pm),Developing (1-3 mo),Sedan,Compact/Midsize,30.0,2.91,1.98,...,11.45,2.29,100,23.0,True,3.25,3.447826,58.77,55.322174,19.011056
2,3,"Atlanta, GA",South,Weekend Lunch (11am-2pm),New (< 1 month),SUV,SUV,24.0,5.82,3.89,...,17.15,3.43,100,23.0,True,3.25,1.533152,24.92,23.386848,4.018359
3,4,Rural Midwest,Midwest,Weekday Lunch (11am-2pm),Developing (1-3 mo),Sedan,Compact/Midsize,30.0,5.99,3.34,...,10.02,1.67,100,23.0,True,3.60,6.079304,22.48,16.400696,2.738013
4,5,"Mountain View, CA",Pacific,Weekday Midday (2pm-5pm),Experienced (1-2 yr),Sedan,Compact/Midsize,30.0,6.46,4.50,...,56.43,5.13,100,23.0,True,4.80,9.635478,113.04,103.404522,16.006892
5,6,"Nashville, TN",South,Weekend Dinner (5pm-8pm),New (< 1 month),Truck,Truck,19.0,7.60,5.56,...,41.16,3.43,100,20.0,True,3.25,9.130875,86.63,77.499125,10.197253
6,7,"Philadelphia, PA",Northeast,Weekday Midday (2pm-5pm),Developing (1-3 mo),Sedan,Compact/Midsize,30.0,2.60,1.72,...,16.35,5.45,100,23.0,True,4.05,2.114804,24.15,22.035196,8.475075
7,8,"Boston, MA",Northeast,Weekday Midday (2pm-5pm),Developing (1-3 mo),Sedan,Compact/Midsize,30.0,6.26,3.77,...,17.99,2.57,100,23.0,True,4.05,6.590935,48.48,41.889065,6.691544
8,9,Suburban South,South,Weekday Dinner (5pm-8pm),Established (3-12 mo),Truck,Truck,19.0,4.39,2.65,...,75.18,10.74,100,20.0,True,3.25,5.346250,106.77,101.423750,23.103360
9,10,Suburban South,South,Weekday Dinner (5pm-8pm),Experienced (1-2 yr),Truck,Truck,19.0,2.15,1.45,...,9.55,1.91,100,20.0,True,3.25,6.763250,21.84,15.076750,7.012442


In [6]:
from google.colab import drive
import os

drive.mount('/content/drive')

OUTPUT_PATH = "/content/drive/MyDrive/DoorDash Project"

# create the directory if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

df.to_csv(f"{OUTPUT_PATH}/earnings_clean.csv", index=False)

print("Dataset saved successfully")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset saved successfully
